In [28]:
import os
import numpy as np
import torch
import pandas as pd
from numpy import kron
import functools as fu
from itertools import *
from itertools import product
from numpy.linalg import cholesky, eig
from qiskit.quantum_info import random_density_matrix
from qutip import Qobj, fidelity
from scipy.stats import norm
from qiskit.quantum_info import random_statevector

In [14]:
!pip list | grep qiskit

qiskit                  2.3.1
qiskit-aer              0.17.2
qiskit-algorithms       0.4.0
qiskit-ibm-runtime      0.49.0
qiskit-qasm3-import     0.6.0


### leading dimension to set

In [ ]:
# number of particles
d = 4

#local dimension (qubit, qutrit...)
local_dim = 2 

 ## Depolarization channel

Here, we provide a local and a global depolarizing channel to introduce an extra physical channel. These files can be used in step 3 during the data generation and linear inversion pre-processing.


The channel that we use for noise in this case is $$\rho_{final} = (1-p)\rho + p/3 (X\rho X^\dagger + Y \rho Y^\dagger + Z \rho Z^\dagger)$$ 

In [ ]:
def local_depolarizing_channel(qubits_number, density_matrix, probability):
	"""
	NOTE: this function must be used inside a for loop to get to apply each pauli's operator for each state
	"""

	pX = np.array([[0.,1.], [1.,0.]]) 			# X Pauli matrix
	pY = np.array([[0.,-1.j], [1.j, 0.]]) 		# Y Pauli matrix
	pZ = np.array([[1., 0.], [0.,-1.]]) 		# Z Pauli matrix
	pI = np.array([[1.,0.], [0.,1.]])
	new_density_matrix = []

	for i in range(qubits_number):
		
		operator_string_x = list(repeat(pI,qubits_number))
		operator_string_y = list(repeat(pI,qubits_number))
		operator_string_z = list(repeat(pI,qubits_number))
		

		operator_string_x[i] = pX
		operator_string_y[i] = pY
		operator_string_z[i] = pZ

		X = fu.reduce(np.kron,operator_string_x)
		Y = fu.reduce(np.kron,operator_string_y)
		Z = fu.reduce(np.kron,operator_string_z)
	
		new_density_matrix = (1-probability)*density_matrix 
		
		new_density_matrix = new_density_matrix + (probability/3)*X.dot(density_matrix).dot(X)
		# print(new_density_matrix.shape)
		new_density_matrix = new_density_matrix + (probability/3)*Y.dot(density_matrix).dot(Y)

		new_density_matrix = new_density_matrix + (probability/3)*Z.dot(density_matrix).dot(Z)
			
	return new_density_matrix


In [17]:
def global_depo_channel(dm,p):
    
    dim = dm.shape[0]
    I = np.eye(dim,dim)
    
    return (1-p)*dm + (p/(dim))*I
    

## Support functions:
1. the vectorization of the Cholesky matrices $vec(C_{ij})$.
2. positive semidefinite brute-force approximation of non physical matrices $\rho_{LI}$(eigenvaluescheck, PureEigenvaluesCheck)


In [18]:

def vectorization_new(rho):
	
	dim = rho.shape[0]
	#chol = np.linalg.cholesky(rho)
	chol = 	torch.linalg.cholesky(torch.complex(torch.Tensor(rho.real),torch.Tensor(rho.imag)))

	diag = np.diag(chol).real.tolist()

	lower_t_indeces = np.tril_indices_from(chol,k=-1)

	reals = list(chol[lower_t_indeces].real.flatten())
	imags = list(chol[lower_t_indeces].imag.flatten())

	return np.array(diag+reals+imags, dtype=np.float64)


def from_mpc_tonumpy(colored_rho):

	"""
	input: a density matrix, mpc type
	output: nd.array of the same density matrix recasted in complex128 type
	"""
	placeholder = np.zeros((colored_rho.shape[0],colored_rho.shape[1]),dtype=complex)
	for r in range(colored_rho.shape[0]):
		for c in range(colored_rho.shape[1]):
			a =np.double(mp.nstr(mp.re(colored_rho[r][c])))
			b = np.double(mp.nstr(mp.im(colored_rho[r][c])))
			built = complex(a,b)
			placeholder[r][c] = built
	return placeholder

def eigenvaluesCheck(dm):

	eis, eigvecs = eig(dm)

	eis[eis.real <0 ] = 0.0001
	eis = [ el.real for el in eis]
	eis = np.array(eis)

	cleanRho = eigvecs@ np.diag(eis) @ eigvecs.T.conj()

	return cleanRho/np.trace(cleanRho)
	
def PureEigenvaluesCheck(dm):

	eis, eigvecs = eig(dm)

	eis[eis.real.round(2) < 0.99 ] = 0.0001
	eis = [ el.real for el in eis]
	eis = np.array(eis)

	cleanRho = eigvecs@ np.diag(eis) @ eigvecs.T.conj()

	return cleanRho/np.trace(cleanRho)

## 3) Random states generation, pre-processing, training dataset preparation.

Here we orderly do:

1. generate random matrices
2. calculate Born values
3. approximate with fine statistics, fixing the parameter "trials" (for SICS), or normalizing the Gaussian noise (for Pauli)
4. using linear inversion to reconstruct the density matrix, and approximate it to the close positive definite approximation thereof, killing the negative eigenvalues
5. vectorize the original matrix and the new, experimental one. This is our training dataset

In [ ]:
# ---------- Pauli / computational basis ----------
I2 = np.eye(2, dtype=complex)
PAULI = {
    'I': np.eye(2, dtype=complex),
    'X': np.array([[0,1],[1,0]], dtype=complex),
    'Y': np.array([[0,-1j],[1j,0]], dtype=complex),
    'Z': np.array([[1,0],[0,-1]], dtype=complex),
}

SINGLE_QUBIT_EIGENBASES = {
    'Z': [np.array([1,0], dtype=complex), np.array([0,1], dtype=complex)],
    'X': [np.array([1,1], dtype=complex)/np.sqrt(2), np.array([1,-1], dtype=complex)/np.sqrt(2)],
    'Y': [np.array([1,1j], dtype=complex)/np.sqrt(2), np.array([1,-1j], dtype=complex)/np.sqrt(2)],
}

def pauli_basis(n, axis='Z'):
    """Return the 2^n computational-type basis vectors for measuring every qubit
    in the same single-qubit Pauli eigenbasis ('X','Y', or 'Z')."""
    single = SINGLE_QUBIT_EIGENBASES[axis]
    basis = [np.array([1.0])]
    for _ in range(n):
        basis = [np.kron(b, s) for b in basis for s in single]
    return basis

def all_pauli_bases(n):
    """Dict of the three standard product Pauli measurement bases for n qubits."""
    return {axis: pauli_basis(n, axis) for axis in ('X','Y','Z')}

def pauli_group(n):
    """Generate the n-qubit Pauli group elements (tensor products of I,X,Y,Z),
    labeled by their Pauli string, e.g. 'XIZ'."""
    labels = [''.join(p) for p in product('IXYZ', repeat=n)]
    # ops = {}
    ops = []
    for lab in labels:
        M = np.array([[1]], dtype=complex)
        for ch in lab:
            M = np.kron(M, PAULI[ch])
        # ops[lab] = M
        ops.append(M)
    return np.array(ops)

# ---------- Bell basis (n even, pairs of qubits) ----------

def bell_basis():
    """The 4 two-qubit Bell states."""
    b00 = np.array([1,0,0,1], dtype=complex)/np.sqrt(2)   # |Phi+>
    b01 = np.array([1,0,0,-1], dtype=complex)/np.sqrt(2)  # |Phi->
    b10 = np.array([0,1,1,0], dtype=complex)/np.sqrt(2)   # |Psi+>
    b11 = np.array([0,1,-1,0], dtype=complex)/np.sqrt(2)  # |Psi->
    return [b00, b01, b10, b11]

def n_qubit_bell_basis(n):
    """Tensor product of n/2 Bell pairs (n must be even) — a maximally-entangled basis."""
    assert n % 2 == 0, "Bell-pair basis needs an even number of qubits"
    bell = bell_basis()
    basis = [np.array([1.0])]
    for _ in range(n//2):
        basis = [np.kron(b, s) for b in basis for s in bell]
    return basis


In [20]:
general_basis = pauli_group(d)
reconstruction_basis = np.copy(general_basis)/(local_dim ** d)

## Random Mixed State Generation

In [21]:
totarr =[]
hs = []
tot = 1
i = 0
li = []
tracevals = []
trials = 1000       # Number of shots

normTrials = np.sqrt(trials/(local_dim**(d)))

folder = './Pauli_data/'
filename = 'Haar4qubitsTrials'+str(trials)+'PAULI.npy'

if not os.path.isdir(folder):
    os.mkdir(folder)


In [ ]:
def HSdist(A,B):
    return np.trace((A-B)@(A-B)).real

hsm = []

for t in range(trials):

    # PURE STATES
    # Rank = 0 (mixed states), Rank = 1 (pure states)
    # 1. Create initial density matrix state
    rho_start0 = np.array(random_density_matrix(local_dim**d, rank=1, method='Hilbert-Schmidt').data)
    rho_start0 = global_depo_channel(rho_start0, 0.1)

    # 2. Measure expectation values of density matrix in Pauli basis
    borns =  np.array([ np.trace(rho_start0 @ base).real for base in general_basis])
    
    # 3. Generates random noise for measurement data
    singleFreqVariance = norm.rvs(size = local_dim**(2*d))/8  

    # 4. Add the noise to the expectation values 
    borns_approx =  [ el1 + el2 for el1, el2 in zip(borns, singleFreqVariance) ]  
    
    # 5. Reconstruct the approximation, first stage of linear inversion
    rback = sum([ np.round(reconstruction_basis[i],12) * borns_approx[i] for i in range(4**d) ])
    
    # 6. Brute-force approximation upon LI
    cleanRho = eigenvaluesCheck(rback)           # Throws out negative eigenvals
    chol_exp = [ vectorization_new(cleanRho)  ]  # Makes Cholesky Decomposition representation - send to NN

    try:
        # 7. Linear Inversion Test
        li.append(fidelity(Qobj(cleanRho),Qobj(rho_start0)))

        # 8. Calculate Hilbert-Schmidt Distance
        hs.append(HSdist(cleanRho,rho_start0 ))
        
        # print(np.trace(rho_start0@rho_start0))

        # 9. Ideal Cholesky
        chol_theoretic = vectorization_new(PureEigenvaluesCheck(rho_start0))

        # 10. Calculate difference between experimental and theoretical Cholesky's
        totarr.append( np.concatenate((chol_exp, chol_theoretic), axis=None) )

    except RuntimeError:
        pass

np.save(folder + filename, np.array(totarr))


In [30]:
np.mean(li)

np.float64(0.8340796250515757)

## Haar-Random Pure State Generation

In [ ]:
hsm = []

for t in range(trials):

    # PURE STATES
    # Rank = 0 (mixed states), Rank = 1 (pure states)
    rho_start0 = np.array(random_statevector(local_dim**d).data)
    rho_start0 = global_depo_channel(rho_start0, 0.1)
    borns =  np.array([ np.trace(rho_start0 @ base).real for base in general_basis])
    
    
    singleFreqVariance = norm.rvs(size = local_dim**(2*d))/8  # Generating random numbers
    
    borns_approx =  [ el1 + el2 for el1, el2 in zip(borns, singleFreqVariance) ]  
    
    #reconstruct the approximation, first stage of linear inversion
    rback = sum([ np.round(reconstruction_basis[i],12) * borns_approx[i] for i in range(4**d) ])
    
    # brute-force approximation upon LI
    cleanRho = eigenvaluesCheck(rback)
    chol_exp = [ vectorization_new(cleanRho)  ]

    try:
        li.append(fidelity(Qobj(cleanRho),Qobj(rho_start0)))
        hs.append(HSdist(cleanRho,rho_start0 ))        

        chol_theoretic = vectorization_new(PureEigenvaluesCheck(rho_start0))
        totarr.append( np.concatenate((chol_exp, chol_theoretic), axis=None) )

    except RuntimeError:
        pass

In [32]:
np.mean(li)

np.float64(0.9134908584603392)

## Random Product State Generation

In [ ]:
hsm = []

for t in range(trials):

    # Generates Random Product States
    rho_start0 = np.array(random_statevector(2).data)
    for a in range(d-1):
        rho_start0 = np.kron(rho_start0, np.array(random_statevector(2).data))

    rho_start0 = global_depo_channel(rho_start0, 0.1)
    borns =  np.array([ np.trace(rho_start0 @ base).real for base in general_basis])

    singleFreqVariance = norm.rvs(size = local_dim**(2*d))/8  # Generating random numbers
    
    borns_approx =  [ el1 + el2 for el1, el2 in zip(borns, singleFreqVariance) ]  
    
    #reconstruct the approximation, first stage of linear inversion
    rback = sum([ np.round(reconstruction_basis[i],12) * borns_approx[i] for i in range(4**d) ])
    
    # brute-force approximation upon LI
    cleanRho = eigenvaluesCheck(rback)
    chol_exp = [ vectorization_new(cleanRho)  ]

    try:
        # Linear Inversion Test
        li.append(fidelity(Qobj(cleanRho),Qobj(rho_start0)))
        hs.append(HSdist(cleanRho,rho_start0 ))
        
        # print(np.trace(rho_start0@rho_start0))

        chol_theoretic = vectorization_new(PureEigenvaluesCheck(rho_start0))
        totarr.append( np.concatenate((chol_exp, chol_theoretic), axis=None) )

    except RuntimeError:
        pass

In [34]:
np.mean(li)

np.float64(0.9656591415730089)